In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 1. Load our clean retail dataset
print("Loading data...")
df = pd.read_csv('online_retail.csv')

# 2. Clean up any rows missing CustomerID or Description to ensure perfect mapping
df = df.dropna(subset=['CustomerID', 'Description'])

# 3. Create the Item-User Pivot Matrix
# Rows = Products (Description), Columns = Customers (CustomerID)
print("Building Item-Customer Matrix (this might take a few seconds)...")
item_user_matrix = df.pivot_table(
    index='Description', 
    columns='CustomerID', 
    values='Quantity', 
    aggfunc='sum'
).fillna(0)  # Fill NaNs with 0 because a missing value means that customer never bought that item

# 4. Display the shape of our new matrix
print("\n--- Matrix Generation Complete! ---")
print(f"Total Unique Products (Rows): {item_user_matrix.shape[0]}")
print(f"Total Unique Customers (Columns): {item_user_matrix.shape[1]}")
print("\n--- Quick Peek at the Matrix Grid ---")
print(item_user_matrix.iloc[:5, :5])

Loading data...
Building Item-Customer Matrix (this might take a few seconds)...

--- Matrix Generation Complete! ---
Total Unique Products (Rows): 4223
Total Unique Customers (Columns): 4372

--- Quick Peek at the Matrix Grid ---
CustomerID                     12346  12347  12348  12349  12350
Description                                                     
4 PURPLE FLOCK DINNER CANDLES    0.0    0.0    0.0    0.0    0.0
50'S CHRISTMAS GIFT BAG LARGE    0.0    0.0    0.0    0.0    0.0
DOLLY GIRL BEAKER                0.0    0.0    0.0    0.0    0.0
I LOVE LONDON MINI BACKPACK      0.0    0.0    0.0    0.0    0.0
I LOVE LONDON MINI RUCKSACK      0.0    0.0    0.0    0.0    0.0


In [2]:
# 1. Compute the Cosine Similarity matrix across all product rows
print("Calculating item similarity scores (this might take a moment)...")
item_similarity = cosine_similarity(item_user_matrix)

# 2. Turn the resulting numpy array into a structured DataFrame
item_similarity_df = pd.DataFrame(
    item_similarity, 
    index=item_user_matrix.index, 
    columns=item_user_matrix.index
)

print("--- Item Similarity Matrix Generated! ---")
print(f"Matrix Dimensions: {item_similarity_df.shape}")

# 3. Take a quick look at a small chunk of the similarity grid
print("\n--- Sample Similarity Scores ---")
print(item_similarity_df.iloc[:5, :5].round(3))

Calculating item similarity scores (this might take a moment)...
--- Item Similarity Matrix Generated! ---
Matrix Dimensions: (4223, 4223)

--- Sample Similarity Scores ---
Description                    4 PURPLE FLOCK DINNER CANDLES  \
Description                                                    
4 PURPLE FLOCK DINNER CANDLES                          1.000   
50'S CHRISTMAS GIFT BAG LARGE                          0.007   
DOLLY GIRL BEAKER                                      0.005   
I LOVE LONDON MINI BACKPACK                            0.026   
I LOVE LONDON MINI RUCKSACK                            0.000   

Description                    50'S CHRISTMAS GIFT BAG LARGE  \
Description                                                    
4 PURPLE FLOCK DINNER CANDLES                          0.007   
50'S CHRISTMAS GIFT BAG LARGE                          1.000   
DOLLY GIRL BEAKER                                      0.007   
I LOVE LONDON MINI BACKPACK                            0.0

In [3]:
# Search the index for any product containing "LONDON" to see its exact spelling
london_products = [item for item in item_similarity_df.index if 'LONDON' in str(item)]
print("--- Exact names found in catalog ---")
print(london_products[:10])

--- Exact names found in catalog ---
[' I LOVE LONDON MINI BACKPACK', ' I LOVE LONDON MINI RUCKSACK', ' SET 2 TEA TOWELS I LOVE LONDON ', 'CARD I LOVE LONDON ', 'DOORMAT I LOVE LONDON', 'I LOVE LONDON BABY GIFT SET', 'I LOVE LONDON BEAKER', 'I LOVE LONDON WALL ART', 'LANDMARK FRAME LONDON BRIDGE ', 'LONDON BUS COFFEE MUG']


In [4]:
def get_top_recommendations(product_name, top_n=5):
    # 1. Check if the product actually exists in our data catalog
    if product_name not in item_similarity_df.index:
        return f"Error: '{product_name}' was not found in the product catalog."
    
    # 2. Extract the similarity scores for this specific item and sort them
    similar_products = item_similarity_df[product_name].sort_values(ascending=False)
    
    # 3. Drop the product itself and convert to a clean DataFrame
    recommendations_df = pd.DataFrame(similar_products.iloc[1:top_n+1]).reset_index()
    recommendations_df.columns = ['Recommended Product', 'Similarity Score']
    
    return recommendations_df

# Let's test the engine immediately with one of your catalog items!
# Strip spaces from the string we are looking up to match the data cleanly
test_product = "I LOVE LONDON MINI BACKPACK"

# Clean up the query string or look for the matching text with the leading space
# We can search dynamically or match the exact item from your list:
exact_match = ' I LOVE LONDON MINI BACKPACK'

print(f"--- Top 5 Recommendations for: '{exact_match.strip()}' ---")
print(get_top_recommendations(exact_match))


--- Top 5 Recommendations for: 'I LOVE LONDON MINI BACKPACK' ---
                Recommended Product  Similarity Score
0      CHILDREN'S APRON DOLLY GIRL           0.903341
1      HOOK, 1 HANGER ,MAGIC GARDEN          0.899749
2                 EASTER TIN BUCKET          0.897228
3   CHARLOTTE BAG DOLLY GIRL DESIGN          0.886017
4  FOLKART ZINC HEART CHRISTMAS DEC          0.881763


In [5]:
def recommend_for_customer(customer_id, top_n=5):
    # 1. Check if customer exists in our matrix
    if customer_id not in item_user_matrix.columns:
        return f"Error: Customer ID {customer_id} not found."
    
    # 2. Find items this customer has already purchased
    customer_history = item_user_matrix[customer_id]
    purchased_items = customer_history[customer_history > 0].index.tolist()
    
    if not purchased_items:
        return "This customer has no purchase history."
    
    # 3. Aggregate similarity scores for ALL items based on what they bought
    total_scores = pd.Series(dtype='float64')
    
    for item in purchased_items:
        # Get how similar every item in the store is to this purchased item
        similarity_weight = item_similarity_df[item]
        # Multiply by the quantity they bought to give more weight to items they love
        quantity_bought = customer_history[item]
        weighted_score = similarity_weight * quantity_bought
        
        # Add it to our running total
        total_scores = total_scores.add(weighted_score, fill_value=0)
        
    # 4. Filter out items they already bought so we don't suggest duplicates
    total_scores = total_scores.drop(index=purchased_items, errors='ignore')
    
    # 5. Sort to find the highest scoring recommendations
    final_recommendations = total_scores.sort_values(ascending=False).head(top_n).reset_index()
    final_recommendations.columns = ['Recommended Product', 'Calculated Score']
    
    return final_recommendations

# Let's test it out on Customer ID 12347 (from your previous matrix output chunk!)
test_user = 12347
print(f"--- Top 5 Personalized Recommendations for Customer {test_user} ---")
print(recommend_for_customer(test_user))

--- Top 5 Personalized Recommendations for Customer 12347 ---
                 Recommended Product  Calculated Score
0        CHARLOTTE BAG PINK POLKADOT       1054.129168
1                  CARD PARTY GAMES        1051.721838
2        RED RETROSPOT CHARLOTTE BAG       1046.936393
3  ROUND SNACK BOXES SET OF 4 SKULLS       1039.829858
4               WALL TIDY RETROSPOT        1037.734760


In [6]:
import pickle
import gzip

# Export the structured similarity DataFrame as a compressed .pkl.gz file
with gzip.open('item_similarity.pkl.gz', 'wb') as f:
    pickle.dump(item_similarity_df, f)
print("Successfully exported compressed item_similarity.pkl.gz!")

Successfully exported compressed item_similarity.pkl.gz!
